In [5]:
from datasets import load_dataset
from helpers import load_qwen_model, get_clip_generator, run_inference, visualize_clip, downsize_clip
import time
import torch

dataset = load_dataset("flwrlabs/ucf101")

# To see what's inside (columns, number of rows, etc.)
print(dataset)

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/79 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'video_id', 'clip_id', 'frame', 'label'],
        num_rows: 1786096
    })
    test: Dataset({
        features: ['image', 'video_id', 'clip_id', 'frame', 'label'],
        num_rows: 697222
    })
})


In [3]:
test_data = dataset["test"]

#press q to quit the video window
for clip in get_clip_generator(test_data, max_rows=10):
    print(f"\n🎬 Clip: {clip['clip_id']} | Expected: {clip['label_name']}")
    
    visualize_clip(clip['images'], fps = 30)  # Optional: visualize the clip frames 


🎬 Clip: v_ApplyEyeMakeup_g01_c01 | Expected: ApplyEyeMakeup
Press 'q' to close the video window.


In [14]:
import os
import time
import pandas as pd
import torch

def run_pipeline(
    dataset_split, 
    model, 
    processor, 
    output_path="results/ucf_captioning_results.csv", 
    max_clips=10000,
    save_every=10,
    prompt="Describe this video."
):
    """
    Runs the Qwen-VL captioning pipeline and saves results incrementally.
    """
    # 1. Setup Storage
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    columns = ["clip_id", "ground_truth", "output", "latency", "num_downsampled_frames", "num_frames"]
    
    # Initialize CSV with header
    pd.DataFrame(columns=columns).to_csv(output_path, index=False)
    
    results_log = []
    total_processed = 0
    start_wall_time = time.time()

    print(f"🚀 Starting Pipeline (Max Clips: {max_clips})...")

    # 2. Iterate through clips using your generator
    for clip in get_clip_generator(dataset_split, max_rows=max_clips):
        try:
            total_processed += 1
            clip_start = time.time()
            
            # Prepare frames
            # Logic: downsize_clip helper used here
            downsized_frames = downsize_clip(clip['images'], fps=1)
            
            # Model Inference
            # Assuming run_inference handles the prompt: "Describe this video."
            response = run_inference(model, processor, downsized_frames, prompt = prompt)
            
            duration = time.time() - clip_start
            
            # Data entry
            result_entry = {
                "clip_id": clip['clip_id'],
                "ground_truth": clip['label_name'],
                "output": response.strip(),
                "latency": duration,
                "num_downsampled_frames": len(downsized_frames),
                "num_frames": len(clip['images'])
            }
            results_log.append(result_entry)

            # Live Feedback
            print(f"[{total_processed}] 🎬 {result_entry['clip_id']} | GT: {result_entry['ground_truth']} | 🤖: {result_entry['output'][:50]}...")

            # 3. Incremental Save
            if total_processed % save_every == 0:
                pd.DataFrame(results_log).to_csv(output_path, mode='a', index=False, header=False)
                results_log = [] # Clear memory
                print(f"💾 Progress saved to {output_path}")

            # Clear VRAM
            torch.cuda.empty_cache()

        except Exception as e:
            print(f"⚠️ Error processing clip {clip.get('clip_id', 'unknown')}: {e}")
            continue

    # 4. Final Cleanup
    if results_log:
        pd.DataFrame(results_log).to_csv(output_path, mode='a', index=False, header=False)
    
    total_time = time.time() - start_wall_time
    print(f"\n✅ Pipeline Complete!")
    print(f"⏱️ Total Wall Time: {total_time/60:.2f} minutes")
    print(f"📂 Results saved to: {output_path}")

# --- EXECUTION ---

# Load model and data once
model, processor = load_qwen_model()
test_data = dataset["test"]

# Run it
run_pipeline(
    dataset_split=test_data,
    model=model,
    processor=processor,
    max_clips=1000
)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

🚀 Starting Pipeline (Max Clips: 1000)...
[1] 🎬 v_ApplyEyeMakeup_g01_c01 | GT: ApplyEyeMakeup | 🤖: A person is applying makeup, specifically eyeliner...
[2] 🎬 v_ApplyEyeMakeup_g01_c02 | GT: ApplyEyeMakeup | 🤖: A person is applying makeup, specifically eyeshado...
[3] 🎬 v_ApplyEyeMakeup_g01_c03 | GT: ApplyEyeMakeup | 🤖: A woman is applying makeup to her face, specifical...
[4] 🎬 v_ApplyEyeMakeup_g01_c04 | GT: ApplyEyeMakeup | 🤖: A woman is applying makeup in a room with a dark-c...
[5] 🎬 v_ApplyEyeMakeup_g01_c05 | GT: ApplyEyeMakeup | 🤖: A woman is applying makeup in front of a camera. S...

✅ Pipeline Complete!
⏱️ Total Wall Time: 0.31 minutes
📂 Results saved to: results/ucf_captioning_results.csv


In [16]:
# With label options
ucf_labels = test_data.features["label"].names 
labels_str = ", ".join(ucf_labels)

my_prompt = (
    f"Analyze this video clip. From the following list, pick the action that "
    f"best describes what is happening: {labels_str}. "
    f"Provide only the label name and nothing else."
)


run_pipeline(
    dataset_split=test_data,
    model=model, 
    output_path="results/ucf_captioning_results_with_labels.csv",
    processor=processor,
    max_clips=len(test_data),
    prompt=my_prompt
)

🚀 Starting Pipeline (Max Clips: 697222)...
[1] 🎬 v_ApplyEyeMakeup_g01_c01 | GT: ApplyEyeMakeup | 🤖: ApplyEyeMakeup...
[2] 🎬 v_ApplyEyeMakeup_g01_c02 | GT: ApplyEyeMakeup | 🤖: ApplyEyeMakeup...
[3] 🎬 v_ApplyEyeMakeup_g01_c03 | GT: ApplyEyeMakeup | 🤖: ApplyEyeMakeup...
[4] 🎬 v_ApplyEyeMakeup_g01_c04 | GT: ApplyEyeMakeup | 🤖: ApplyEyeMakeup...
[5] 🎬 v_ApplyEyeMakeup_g01_c05 | GT: ApplyEyeMakeup | 🤖: ApplyEyeMakeup...
[6] 🎬 v_ApplyEyeMakeup_g01_c06 | GT: ApplyEyeMakeup | 🤖: ApplyEyeMakeup...
[7] 🎬 v_ApplyEyeMakeup_g02_c01 | GT: ApplyEyeMakeup | 🤖: ApplyEyeMakeup...
[8] 🎬 v_ApplyEyeMakeup_g02_c02 | GT: ApplyEyeMakeup | 🤖: ApplyEyeMakeup...
[9] 🎬 v_ApplyEyeMakeup_g02_c03 | GT: ApplyEyeMakeup | 🤖: ApplyEyeMakeup...
[10] 🎬 v_ApplyEyeMakeup_g02_c04 | GT: ApplyEyeMakeup | 🤖: ApplyEyeMakeup...
💾 Progress saved to results/ucf_captioning_results_with_labels.csv
[11] 🎬 v_ApplyEyeMakeup_g03_c01 | GT: ApplyEyeMakeup | 🤖: ApplyEyeMakeup...
[12] 🎬 v_ApplyEyeMakeup_g03_c02 | GT: ApplyEyeMakeup | 🤖: Apply

In [12]:
print(my_prompt)

Analyze this video clip. From the following list, pick the action that best describes what is happening: ApplyEyeMakeup, ApplyLipstick, Archery, BabyCrawling, BalanceBeam, BandMarching, BaseballPitch, Basketball, BasketballDunk, BenchPress, Biking, Billiards, BlowDryHair, BlowingCandles, BodyWeightSquats, Bowling, BoxingPunchingBag, BoxingSpeedBag, BreastStroke, BrushingTeeth, CleanAndJerk, CliffDiving, CricketBowling, CricketShot, CuttingInKitchen, Diving, Drumming, Fencing, FieldHockeyPenalty, FloorGymnastics, FrisbeeCatch, FrontCrawl, GolfSwing, Haircut, HammerThrow, Hammering, HandstandPushups, HandstandWalking, HeadMassage, HighJump, HorseRace, HorseRiding, HulaHoop, IceDancing, JavelinThrow, JugglingBalls, JumpRope, JumpingJack, Kayaking, Knitting, LongJump, Lunges, MilitaryParade, Mixing, MoppingFloor, Nunchucks, ParallelBars, PizzaTossing, PlayingCello, PlayingDaf, PlayingDhol, PlayingFlute, PlayingGuitar, PlayingPiano, PlayingSitar, PlayingTabla, PlayingViolin, PoleVault, Pomm

In [30]:
import plotly.express as px
import plotly.io as pio

import matplotlib.pyplot as plt

df = pd.read_csv("results/ucf_captioning_results_with_labels.csv")

accuracy_count = (df['ground_truth'] == df['output']).sum()
total_count = len(df)
accuracy = accuracy_count / total_count * 100
print(f"📊 Final Accuracy: {accuracy_count}/{total_count} = {accuracy:.4f}%")
print(f"Average Latency: {df['latency'].mean():.2f} seconds")


confusion_matrix = pd.crosstab(df['ground_truth'], df['output'], rownames=['Actual'], colnames=['Predicted'])


# 1. Create the interactive heatmap
fig = px.imshow(
    confusion_matrix,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=confusion_matrix.columns,
    y=confusion_matrix.index,
    color_continuous_scale='Blues',
    aspect="auto"
)

# 2. Add hover information and clean up the layout
fig.update_layout(
    title='Interactive UCF101 Confusion Matrix',
    xaxis_nticks=101, # Ensure all classes are represented
    yaxis_nticks=101,
    width=1200,       # Initial viewing width
    height=1000       # Initial viewing height
)

# 3. Show the plot
pio.renderers.default = "browser"
fig.show()

📊 Final Accuracy: 2493/3783 = 65.9001%
Average Latency: 1.35 seconds
